# Wine Quality Classification — POSTECH Tech Challenge Fase 2
**Dataset:** WineQT | **Objetivo:** Classificação binária da qualidade do vinho  
**Abordagem:** Pipeline completo — EDA → Pré-processamento → Modelagem → Avaliação → Interpretação

> **Critério binário (conforme enunciado):**  
> - `Alta Qualidade` → nota ≥ 7 → `label = 1`  
> - `Baixa/Média Qualidade` → nota < 7 → `label = 0`

## Imports e Configurações Globais

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
# ImbPipeline: necessário para incluir SMOTE dentro da validação cruzada.
# O Pipeline padrão do sklearn NÃO suporta etapas de resampling.

warnings.filterwarnings('ignore')

# ── Paleta de cores padronizada ───────────────────────────────────────────────
BG_FIG  = '#ffffff'
BG_AX   = '#ffffff'
TEXT    = '#222222'
C_LOW   = '#800000'   # vinho escuro → classe Baixa/Média (0)
C_HIGH  = '#003300'   # verde escuro → classe Alta (1)
C_SPINE = '#aaaaaa'
C_LR    = '#800000'   # Logistic Regression
C_RF    = '#003300'   # Random Forest
C_GB    = '#8B4513'   # Gradient Boosting

def style_ax(ax):
    """Aplica estilo padrão a um eixo matplotlib."""
    ax.set_facecolor(BG_AX)
    ax.tick_params(colors=TEXT, labelsize=12)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(C_SPINE)

os.makedirs('../results', exist_ok=True)
os.makedirs('../data',    exist_ok=True)

print("Bibliotecas carregadas!")
print("Pastas '../results' e '../data' verificadas.")

---
# ETAPA 1 — Compreensão do Problema

## 1.1 Contexto do Negócio

A avaliação sensorial de vinhos é **subjetiva, lenta e cara**, depende de especialistas treinados
e pode variar conforme o avaliador. Variáveis físico-químicas mensuráveis objetivamente durante a produção
oferecem uma alternativa: se conseguirmos prever a qualidade com base nessas métricas, enólogos e
produtores podem **ajustar o processo produtivo antes do produto ser avaliado**, reduzindo retrabalho e
garantindo padronização.

## 1.2 Variável Alvo e Critério de Classificação

| Variável | Tipo | Papel |
|---|---|---|
| `quality` | Inteiro (3–8) | Nota sensorial atribuída por especialistas — **alvo original** |
| `quality_label` | Binário (0/1) | **Alvo do modelo** — 1 se quality ≥ 7, 0 caso contrário |

O corte em 7 é o **critério exato do enunciado** e faz sentido técnico:
notas 7 e 8 representam vinhos que passariam de "aceitável" para "premium" no mercado.

## 1.3 Glossário das Features

| Feature | Unidade | Significado enológico |
|---|---|---|
| `fixed acidity` | g/dm³ | Ácidos não voláteis — contribuem para estrutura e frescor |
| `volatile acidity` | g/dm³ | Ácido acético — em excesso causa sabor de vinagre |
| `citric acid` | g/dm³ | Frescor, complexidade e sabor cítrico |
| `residual sugar` | g/dm³ | Açúcar restante após fermentação |
| `chlorides` | g/dm³ | Teor de sal — afeta sabor e palatabilidade |
| `free sulfur dioxide` | mg/dm³ | SO₂ livre — conservante ativo e antioxidante |
| `total sulfur dioxide` | mg/dm³ | SO₂ total (livre + ligado inativo) |
| `density` | g/cm³ | Massa específica — relacionada ao teor alcoólico e açúcar |
| `pH` | — | Acidez/alcalinidade geral (0=ácido, 14=alcalino) |
| `sulphates` | g/dm³ | Contribui para SO₂ livre; propriedades antimicrobianas |
| `alcohol` | % vol. | Teor alcoólico — forte preditor de qualidade |

In [ ]:
# ============================================================
# 1.4 CARREGAMENTO DOS DADOS — com fallback local
# ============================================================
# Tenta carregar de ../data/ (execução local) antes de baixar da URL.
# Garante reprodutibilidade mesmo sem acesso à internet.

DATA_PATH = os.path.join('..', 'data', 'WineQT.csv')
URL = (
    "https://raw.githubusercontent.com/gioandradebonucci/"
    "Classificao_da_qualidade_dos_vinhos_usando_ML/refs/heads/main/data/WineQT.csv"
)

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset carregado localmente: {DATA_PATH}")
else:
    df = pd.read_csv(URL)
    df.to_csv(DATA_PATH, index=False)
    print(f"Dataset baixado da URL e salvo em {DATA_PATH} para uso futuro.")

df = df.drop(columns=['Id'])   # ID sequencial — sem valor preditivo

print(f"\nShape inicial: {df.shape[0]} amostras × {df.shape[1]} variáveis")
print(f"\nTipos de dados:\n{df.dtypes}")
print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
print(f"\nLinhas duplicadas (exatas): {df.duplicated().sum()}")
print(f"\nEstatísticas descritivas:\n{df.describe().T.round(3).to_string()}")

print("""
─────────────────────────────────────────────────────────────
SO WHAT — Qualidade do Dataset
─────────────────────────────────────────────────────────────
→ Todas as 11 features são numéricas contínuas — sem necessidade de encoding.
→ Valores nulos: zero — nenhuma imputação necessária.
→ Há duplicatas exatas — serão removidas na próxima célula.
→ chlorides, residual sugar e total SO₂ têm amplitude muito grande
  → assimetria severa tratada com log1p no pré-processamento (Etapa 3).
─────────────────────────────────────────────────────────────""")

In [ ]:
# ============================================================
# 1.4b REMOÇÃO DE DUPLICATAS
# ============================================================
# Duplicatas causam: overfitting, peso desproporcional no treino
# e violação de independência entre observações.

n_antes = len(df)
df = df.drop_duplicates(keep='first')
print(f"Duplicatas removidas: {n_antes - len(df)} | Dataset final: {len(df)} linhas")

# ============================================================
# 1.5 TRANSFORMAÇÃO BINÁRIA DA VARIÁVEL ALVO
# ============================================================
df['quality_label'] = (df['quality'] >= 7).astype(int)
df['quality_class']  = df['quality_label'].map({0: 'Baixa/Média (< 7)', 1: 'Alta (≥ 7)'})

binary_counts = df['quality_label'].value_counts()
pct_alta  = binary_counts[1] / len(df) * 100
pct_baixa = binary_counts[0] / len(df) * 100
ratio     = binary_counts[0] / binary_counts[1]

print(f"""
============================================================
  CLASSES APÓS TRANSFORMAÇÃO BINÁRIA
============================================================
  Baixa/Média (0): {binary_counts[0]:>5} amostras ({pct_baixa:.1f}%)
  Alta (1):        {binary_counts[1]:>5} amostras ({pct_alta:.1f}%)
  Razão de desbalanceamento: {ratio:.1f}:1
============================================================

SO WHAT — Desbalanceamento
→ Apenas {pct_alta:.1f}% dos vinhos são Alta Qualidade → razão {ratio:.1f}:1.
→ Modelo naive (sempre "Baixa") teria {pct_baixa:.1f}% de acurácia com F1=0.
→ Estratégia: SMOTE no treino + F1-Score e PR-AUC como métricas primárias.""")

In [ ]:
# ============================================================
# 1.6 GRÁFICO — Distribuição da Variável Alvo
# ============================================================
counts = df['quality'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG_FIG)

cores_bar = [C_LOW if q < 7 else C_HIGH for q in counts.index]
bars = ax1.bar(counts.index, counts.values, color=cores_bar, edgecolor='white', width=0.7)
ax1.bar_label(bars, fontsize=13, fontweight='bold', color=TEXT)
ax1.set(xlabel='Nota de Qualidade', ylabel='Número de Amostras',
        title='Distribuição Original (Escala 3–8)', xticks=counts.index)
ax1.legend(handles=[
    mpatches.Patch(color=C_LOW,  label='Baixa/Média (< 7)'),
    mpatches.Patch(color=C_HIGH, label='Alta (≥ 7)')
], fontsize=12)
style_ax(ax1)

_, _, autotexts = ax2.pie(
    [binary_counts[0], binary_counts[1]],
    labels=['Baixa/Média\n(< 7)', 'Alta\n(≥ 7)'],
    colors=[C_LOW, C_HIGH], autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.6, edgecolor='white'))
[at.set(fontsize=13, fontweight='bold', color='white') for at in autotexts]
ax2.add_artist(plt.Circle((0, 0), 0.40, fc=BG_AX))
ax2.text(0, 0, f'n={len(df)}', ha='center', va='center',
         fontsize=13, fontweight='bold', color=TEXT)
ax2.set_title('Classificação Binária — Variável Alvo', fontsize=13, fontweight='bold')

plt.suptitle('Etapa 1 — Variável Alvo: Distribuição e Binarização',
             fontsize=15, fontweight='bold', y=1.02, color=TEXT)
plt.tight_layout()
plt.savefig('../results/01_variavel_alvo.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

In [ ]:
# ============================================================
# 1.7 TABELA — Médias por Classe e Impacto Diferencial
# ============================================================
# Identifica quais features têm maior variação % entre classes.
# Quanto maior |% Variação|, maior o poder discriminante da feature.

features_all = [c for c in df.columns if c not in ['quality', 'quality_label', 'quality_class']]

summary = df.groupby('quality_class')[features_all].mean().T
summary.columns.name = None
summary['Diferença (Alta − Baixa)'] = summary['Alta (≥ 7)'] - summary['Baixa/Média (< 7)']
summary['% Variação'] = (
    summary['Diferença (Alta − Baixa)'] / summary['Baixa/Média (< 7)'] * 100
).round(1)

print("Médias por classe (ordenado por |% Variação|):")
print(summary.round(4).sort_values('% Variação', key=abs, ascending=False).to_string())
print("""
─────────────────────────────────────────────────
SO WHAT
→ % Variação POSITIVA: feature maior nos vinhos Alta → contribui positivamente.
→ % Variação NEGATIVA: feature menor nos vinhos Alta → presença prejudicial.

TOP DISCRIMINANTES:
→ citric acid (+54%):      frescor — muito maior nos vinhos Alta.
→ volatile acidity (−29%): menos vinagre nos vinhos Alta.
→ sulphates (+16%):        melhor conservação antimicrobiana.
→ alcohol (+13%):          maior teor alcoólico — preditor forte.
→ pH (−0.9%) e density (−0.1%): variação quase nula — menor poder.
─────────────────────────────────────────────────""")

---
# ETAPA 2 — Análise Exploratória de Dados (EDA)

**Objetivos desta etapa:**
1. Investigar a distribuição de cada feature (assimetria, outliers, formato)
2. Comparar distribuições entre classes Alta e Baixa/Média (boxplots)
3. Analisar correlações — Pearson vs Spearman — e multicolinearidade
4. Validar estatisticamente as diferenças entre classes (Mann-Whitney U)

In [ ]:
# ============================================================
# 2.1 DISTRIBUIÇÃO DAS FEATURES — Histograma + KDE
# ============================================================
# KDE suaviza o histograma em curva contínua.
# skew (assimetria): |s|<0.5 normal | 0.5–1 moderada | >1 severa → log1p

features = [c for c in df.columns if c not in ['quality', 'quality_label', 'quality_class']]

fig, axes = plt.subplots(3, 4, figsize=(18, 12), facecolor=BG_FIG)
for ax, feat in zip(axes.flat, features):
    data = df[feat]
    style_ax(ax)
    ax.hist(data, bins=30, color=C_LOW, alpha=0.6, density=True, edgecolor='white', linewidth=0.3)
    data.plot.kde(ax=ax, color=C_HIGH, linewidth=2)
    ax.axvline(data.mean(),   color=C_LOW,  linestyle='--', linewidth=1.5, label=f'Média: {data.mean():.2f}')
    ax.axvline(data.median(), color=TEXT,   linestyle=':',  linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.set(title=f'{feat}\nskew = {data.skew():.2f}', ylabel='Densidade')
    ax.legend(fontsize=10, facecolor='#eeeeee', labelcolor=TEXT)

axes.flat[-1].set_visible(False)
plt.suptitle('EDA — Distribuição das Features (histograma + KDE | skew = assimetria)',
             fontsize=15, fontweight='bold', color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig('../results/02_distribuicao_features.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print(f"{'Feature':<30} {'Skew':>8} {'Diagnóstico'}")
print("─" * 60)
for feat in features:
    s = df[feat].skew()
    diag = "normal" if abs(s) < 0.5 else ("moderada" if abs(s) < 1 else "severa → log1p")
    print(f"{feat:<30} {s:>8.3f}  {diag}")

In [ ]:
# ============================================================
# 2.2 BOXPLOTS POR CLASSE — Separação visual entre grupos
# ============================================================
# Mediana (linha), IQR (caixa 25–75%), bigodes (1.5×IQR), pontos = outliers.

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.patch.set_facecolor(BG_FIG)

for i, feat in enumerate(features):
    ax = axes.flat[i]
    style_ax(ax)
    data_low  = df[df['quality_label'] == 0][feat]
    data_high = df[df['quality_label'] == 1][feat]

    bp = ax.boxplot(
        [data_low, data_high], patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(color=C_SPINE), capprops=dict(color=C_SPINE),
        flierprops=dict(marker='o', markerfacecolor=C_SPINE, markersize=3, alpha=0.4)
    )
    bp['boxes'][0].set_facecolor(C_LOW);  bp['boxes'][0].set_alpha(0.8)
    bp['boxes'][1].set_facecolor(C_HIGH); bp['boxes'][1].set_alpha(0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Baixa/Média\n(< 7)', 'Alta\n(≥ 7)'], fontsize=12, color=TEXT)
    ax.set_title(feat, fontsize=13, fontweight='bold', color=TEXT)

axes.flat[-1].set_visible(False)
plt.suptitle('EDA — Boxplots por Classe (pontos = outliers pelo método IQR)',
             fontsize=15, fontweight='bold', color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig('../results/03_boxplots_por_classe.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print(f"{'Feature':<30} {'Outliers':>10} {'%':>8}  Tratamento")
print("─" * 65)
for feat in features:
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR = Q3 - Q1
    n   = ((df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)).sum()
    pct = n / len(df) * 100
    trat = "RobustScaler mitiga (>5%)" if pct > 5 else "RobustScaler suficiente"
    print(f"{feat:<30} {n:>10} {pct:>7.1f}%  {trat}")
print("Outliers NÃO removidos — representam variações físico-químicas reais.")

In [ ]:
# ============================================================
# 2.3 MAPA DE CORRELAÇÃO — Pearson (triângulo inferior)
# ============================================================
# Detecta: (a) features correlacionadas com quality
#          (b) pares com |r| > 0.60 → risco de multicolinearidade

corr_matrix = df[features + ['quality']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10), facecolor=BG_FIG)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap=sns.diverging_palette(10, 130, s=80, l=40, as_cmap=True),
    center=0, vmin=-1, vmax=1, linewidths=0.5, linecolor='#dddddd',
    annot_kws={'size': 11, 'fontweight': 'bold'}, cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set(facecolor=BG_AX, title='Mapa de Correlação — Pearson (triângulo inferior)')
ax.title.set(fontsize=14, fontweight='bold', color=TEXT)
ax.tick_params(colors=TEXT, labelsize=12)
plt.tight_layout()
plt.savefig('../results/04_mapa_correlacao.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print("Pares com |correlação| > 0.60 (risco de multicolinearidade):")
print("─" * 70)
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.60:
            print(f"  {corr_matrix.columns[i]:<30} ↔  {corr_matrix.columns[j]:<30}  r = {val:+.3f}")

print("""
Decisão: MANTER todas as features porque:
1. Modelos principais (RF, GB) são baseados em árvores — multicolinearidade impacta menos
2. Dataset pequeno (1.018 amostras) — cada feature é valiosa
3. Correlações < 0.70 → ainda há informação única em cada par""")

In [ ]:
# ============================================================
# 2.4 RANKING DE CORRELAÇÃO COM QUALITY — Pearson
# ============================================================
corr_quality = df[features + ['quality']].corr()['quality'].drop('quality').sort_values()
colors_rank  = [C_LOW if v < 0 else C_HIGH for v in corr_quality.values]

fig, ax = plt.subplots(figsize=(13, 6), facecolor=BG_FIG)
style_ax(ax)
bars = ax.barh(corr_quality.index, corr_quality.values, color=colors_rank, edgecolor='white')
for bar, val in zip(bars, corr_quality.values):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height() / 2, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', color=TEXT, fontsize=11)

ax.axvline(0,     color=C_SPINE,   linewidth=1.0)
ax.axvline( 0.20, color='#888888', linewidth=0.8, linestyle='--', label='Limiar ±0.20')
ax.axvline(-0.20, color='#888888', linewidth=0.8, linestyle='--')
ax.set_xlabel('Correlação de Pearson com quality', fontsize=13, color=TEXT)
ax.set_title('Ranking de Correlação com Quality\n'
             '(vermelho = negativa | verde = positiva | tracejado = ±0.20)',
             fontsize=14, fontweight='bold', color=TEXT)
ax.legend(facecolor='#eeeeee', labelcolor=TEXT, fontsize=10)
plt.tight_layout()
plt.savefig('../results/05_correlacao_quality.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print("""
  TOP PREDITORES POSITIVOS  (correlação > +0.20):
     alcohol (+0.485)    — FORTE — maior teor alcoólico = maior qualidade
     sulphates (+0.258)  — MODERADA — melhor conservação antimicrobiana
     citric acid (+0.242)— MODERADA — frescor e complexidade

  TOP PREDITORES NEGATIVOS (correlação < −0.18):
     volatile acidity (−0.409) — FORTE — excesso causa sabor de vinagre
     density (−0.185)          — FRACA
     total SO₂ (−0.182)        — FRACA

  PREDITORES FRACOS (|r| < 0.15):
     chlorides, free SO₂, pH, residual sugar
     → Úteis em combinações (Feature Engineering — Etapa 3)""")

In [ ]:
# ============================================================
# 2.5 TESTE ESTATÍSTICO — Mann-Whitney U por Feature
# ============================================================
# Por que Mann-Whitney e não t-test?
# → t-test assume distribuição normal (várias features são assimétricas)
# → Mann-Whitney é NÃO-PARAMÉTRICO: não assume normalidade
#
# H0: as duas classes têm a mesma distribuição para a feature.
# Se p-value < 0.05 → rejeitamos H0 → feature discrimina as classes.
#
# Effect size (rank-biserial r):
#   |r| < 0.1 → negligenciável | 0.1–0.3 → pequeno | 0.3–0.5 → médio | >0.5 → grande

low_group  = df[df['quality_label'] == 0]
high_group = df[df['quality_label'] == 1]
n1, n2     = len(low_group), len(high_group)

print(f"{'Feature':<30} {'p-value':>12} {'Effect |r|':>12} {'Magnitude':>15} {'Discrimina?':>13}")
print("─" * 88)

for feat in features:
    u_stat, p_val = stats.mannwhitneyu(low_group[feat], high_group[feat], alternative='two-sided')
    r   = abs(1 - (2 * u_stat) / (n1 * n2))
    mag = "Grande" if r >= 0.5 else ("Médio" if r >= 0.3 else ("Pequeno" if r >= 0.1 else "Negligenciável"))
    sig = "Sim" if p_val < 0.05 else "Não"
    print(f"{feat:<30} {p_val:>12.4f} {r:>12.3f} {mag:>15} {sig:>13}")

print("""
→ Features com p < 0.05 E |r| > 0.30 são as mais discriminantes.
→ Mesmo features fracas são mantidas — o modelo as pondera automaticamente.""")

# ============================================================
# 2.6 PEARSON vs SPEARMAN — Detectando relações não-lineares
# ============================================================
# Pearson: correlação LINEAR. Spearman: baseada em ranks (monotônica).
# Diferença grande → relação não-linear ou outliers inflando Pearson.

print(f"\n{'Feature':<30} {'Pearson':>10} {'Spearman':>10} {'Diferença':>12} {'Tipo'}")
print("─" * 80)

comparativo = {}
for feat in features:
    r_p, _ = pearsonr(df[feat],  df['quality'])
    r_s, _ = spearmanr(df[feat], df['quality'])
    diff   = r_s - r_p
    tipo   = "Linear" if abs(diff) < 0.05 else ("Não-linear (monotônica)" if abs(r_s) > abs(r_p) else "Outliers inflando r")
    comparativo[feat] = {'pearson': r_p, 'spearman': r_s, 'diff': diff}
    print(f"{feat:<30} {r_p:>10.3f} {r_s:>10.3f} {diff:>+12.3f}  {tipo}")

# Gráfico comparativo
feats_ord = sorted(comparativo, key=lambda x: comparativo[x]['spearman'])
x_pos = np.arange(len(feats_ord)); width = 0.35

fig, ax = plt.subplots(figsize=(13, 6), facecolor=BG_FIG)
style_ax(ax)
ax.bar(x_pos - width/2, [comparativo[f]['pearson']  for f in feats_ord],
       width, label='Pearson',  color=C_LOW,  alpha=0.8, edgecolor='white')
ax.bar(x_pos + width/2, [comparativo[f]['spearman'] for f in feats_ord],
       width, label='Spearman', color=C_HIGH, alpha=0.8, edgecolor='white')

ax.set_xticks(x_pos)
ax.set_xticklabels(feats_ord, rotation=35, ha='right', fontsize=11, color=TEXT)
ax.axhline(0,     color=C_SPINE,   linewidth=1.0)
ax.axhline( 0.20, color='#888888', linewidth=0.8, linestyle='--', alpha=0.7)
ax.axhline(-0.20, color='#888888', linewidth=0.8, linestyle='--', alpha=0.7)
ax.set_ylabel('Coeficiente de Correlação com Quality', fontsize=13, color=TEXT)
ax.set_title('Pearson vs Spearman — Correlação com Quality\n(diferença grande = relação não-linear)',
             fontsize=14, fontweight='bold', color=TEXT)
ax.legend(fontsize=12, facecolor='#eeeeee', labelcolor=TEXT)
plt.tight_layout()
plt.savefig('../results/05b_pearson_vs_spearman.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print("""
SO WHAT — Pearson vs Spearman
→ alcohol e volatile acidity: Spearman ≈ Pearson → relação linear.
→ citric acid e sulphates: Spearman > Pearson → relação não-linear.
   Modelos baseados em árvores (RF, GB) têm vantagem aqui.
→ residual sugar e free SO₂: Spearman muito maior → outliers distorciam Pearson.
→ Conclusão: componentes não-lineares relevantes → justifica RF e GB sobre LR.""")

---
# ETAPA 3 — Pré-processamento de Dados

**Pipeline adotado (ordem importa!):**

```
1. log1p         → features com |skew| > 1 (reduz assimetria)
2. Feature Eng.  → 3 razões enológicas combinadas
3. RobustScaler  → normalização resistente a outliers
4. Train/Test    → 80/20 estratificado
5. SMOTE         → somente no treino (sem data leakage)
```

| Método | Fórmula | Sensível a Outliers | Adequado? |
|---|---|---|---|
| StandardScaler | (x − média) / std | Sim | Não |
| MinMaxScaler | (x − mín) / (máx − mín) | Extremamente | Não |
| **RobustScaler** | **(x − mediana) / IQR** | Não | **Sim** |

| Balanceamento | Como funciona | Problema |
|---|---|---|
| Random Oversampling | Duplica amostras | Overfitting |
| Undersampling | Remove da classe maior | Perde dados reais |
| **SMOTE** | Interpola k-vizinhos | Diversidade sem perda |

In [ ]:
# ============================================================
# 3.1 TRANSFORMAÇÃO LOG1P — Reduz assimetria severa
# ============================================================
# log1p(x) = log(1+x): seguro para x=0 (evita log(0) = -inf)
# Benefício: melhora modelos lineares E gera amostras SMOTE mais realistas

skewed_features = [
    'residual sugar',       # skew severo
    'chlorides',            # skew severo
    'sulphates',            # moderado-alto
    'free sulfur dioxide',  # assimétrico à direita
    'total sulfur dioxide', # assimétrico à direita
    'fixed acidity'         # moderado
]

df_proc = df.copy()

print(f"{'Feature':<30} {'Skew Antes':>12} {'Skew Depois':>12} {'Redução':>10}")
print("─" * 70)
for feat in skewed_features:
    before = df_proc[feat].skew()
    df_proc[feat + '_log'] = np.log1p(df_proc[feat])
    after  = df_proc[feat + '_log'].skew()
    reducao = (1 - abs(after) / abs(before)) * 100
    print(f"{feat:<30} {before:>12.3f} {after:>12.3f} {reducao:>9.1f}%")

In [ ]:
# ============================================================
# 3.2 FEATURE ENGINEERING — Razões com racional enológico
# ============================================================

# acid_ratio: equilíbrio entre acidez estrutural e ácido acético
df_proc['acid_ratio']      = df_proc['fixed acidity'] / (df_proc['volatile acidity'] + 0.001)

# so2_ratio: eficiência do conservante (fração ativa do SO₂)
df_proc['so2_ratio']       = df_proc['free sulfur dioxide'] / (df_proc['total sulfur dioxide'] + 0.001)

# alcohol_density: proxy da completude da fermentação
df_proc['alcohol_density'] = df_proc['alcohol'] / df_proc['density']

new_features = ['acid_ratio', 'so2_ratio', 'alcohol_density']

print("Correlação Spearman com quality — Originais vs Engineered:")
print("─" * 60)
for feat in ['fixed acidity', 'volatile acidity', 'free sulfur dioxide',
             'total sulfur dioxide', 'alcohol', 'density']:
    r = spearmanr(df_proc[feat], df_proc['quality'])[0]
    print(f"  {feat:<32}: {r:+.4f}")

print()
for feat in new_features:
    r = spearmanr(df_proc[feat], df_proc['quality'])[0]
    print(f"  {feat:<32}: {r:+.4f}  ← engineered")

In [ ]:
# ============================================================
# 3.3 NORMALIZAÇÃO — RobustScaler
# ============================================================
features_model = [
    (feat + '_log' if feat in skewed_features else feat)
    for feat in features
] + new_features

X = df_proc[features_model].copy()
y = df_proc['quality_label'].copy()

scaler   = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features_model)

print(f"Features no modelo: {len(features_model)}")
print(f"  {features_model}")

# ============================================================
# 3.4 DIVISÃO TREINO/TESTE + SMOTE
# ============================================================
# Estratificado: mantém proporção 86:14 em treino e teste.
# SMOTE SOMENTE no treino — aplicar antes seria data leakage.

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nTreino original: {X_train.shape[0]} | Teste: {X_test.shape[0]}")
print(f"Treino após SMOTE: {X_train_bal.shape[0]}")
print(f"  Baixa/Média: {(y_train_bal==0).sum()} | Alta: {(y_train_bal==1).sum()}")

---
# ETAPA 4 — Desenvolvimento dos Modelos

## 4.1 Modelos e Justificativas

| Modelo | Tipo | Vantagem | Limitação |
|---|---|---|---|
| **Logistic Regression** | Linear | Interpretável, baseline claro | Não captura não-linearidades |
| **Random Forest** | Bagging | Robusto, reduz variância | Menos interpretável |
| **Gradient Boosting** | Boosting | Alta precisão, aprende erros iterativamente | Sensível a hiperparâmetros |

## 4.2 Métricas e Justificativas

Com **desbalanceamento 6:1**, Accuracy é enganosa — modelo naive atinge 86% com F1=0.

| Métrica | Por que importa aqui |
|---|---|
| **F1-Score** | Métrica principal — penaliza desequilíbrio |
| **PR-AUC** | Mais honesta que ROC-AUC com desbalanceamento |
| **Recall** | Custo de perder um vinho premium (falso negativo) |
| **Precision** | Custo de classificar vinho comum como premium |
| ROC-AUC | Complementar — independente do limiar |
| Accuracy | Reportada para completude, **NÃO** é a principal |

In [ ]:
# ============================================================
# 4.3 TREINAMENTO DOS MODELOS
# ============================================================

# Logistic Regression — baseline linear
# class_weight='balanced': penaliza erros na classe Alta proporcionalmente
lr = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_bal, y_train_bal)

# Random Forest — 200 árvores com bagging
# min_samples_leaf=2: evita folhas com 1 amostra (overfitting)
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf.fit(X_train_bal, y_train_bal)

# Gradient Boosting — boosting sequencial
# lr=0.05 + n_estimators=300: passos pequenos, muitos deles → menos overfitting
# subsample=0.8: stochastic boosting — regulariza sem perder dados
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=42
)
gb.fit(X_train_bal, y_train_bal)

nomes = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']
cores = [C_LR, C_RF, C_GB]

print("Modelos treinados com sucesso:")
print(f"  LR : C=1.0, max_iter=1000, class_weight=balanced")
print(f"  RF : {rf.n_estimators} árvores, min_samples_leaf={rf.min_samples_leaf}")
print(f"  GB : {gb.n_estimators} est., lr={gb.learning_rate}, depth={gb.max_depth}")

In [ ]:
# ============================================================
# 4.4 VALIDAÇÃO CRUZADA ESTRATIFICADA — SMOTE dentro de cada fold
# ============================================================
# ImbPipeline: garante SMOTE apenas no treino de cada fold.
# Pipeline sklearn padrão NÃO suporta resampling.
# Sem ImbPipeline → data leakage (amostras sintéticas do teste contaminam a avaliação).

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_pipeline = {
    'Logistic Regression': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                         min_samples_leaf=2, random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                              max_depth=4, subsample=0.8, random_state=42))
    ])
}

print("Validação Cruzada Estratificada — 5-fold (SMOTE dentro de cada fold)")
print("─" * 75)
print(f"{'Modelo':<25} {'F1 Médio':>10} {'F1 ±std':>10} {'AUC Médio':>10} {'AUC ±std':>10}")
print("─" * 75)

cv_results = {}
for nome, pipeline in modelos_pipeline.items():
    f1_cv  = cross_val_score(pipeline, X_scaled, y, cv=cv, scoring='f1',      n_jobs=-1)
    auc_cv = cross_val_score(pipeline, X_scaled, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[nome] = {'f1': f1_cv, 'auc': auc_cv}
    print(f"{nome:<25} {f1_cv.mean():>10.4f} {f1_cv.std():>10.4f}"
          f" {auc_cv.mean():>10.4f} {auc_cv.std():>10.4f}")

# Interpretação dinâmica (sem valores hardcoded)
melhor_cv = max(cv_results, key=lambda m: cv_results[m]['f1'].mean())
print(f"""
INTERPRETAÇÃO DA VALIDAÇÃO CRUZADA:
→ Melhor F1 médio: {melhor_cv} ({cv_results[melhor_cv]['f1'].mean():.4f} ± {cv_results[melhor_cv]['f1'].std():.4f})
→ GB tem menor variação → mais estável entre diferentes amostras.
→ AUC alto para todos (> 0.86) → todos separam bem as classes.
→ LR confirma limitação linear: F1 médio inferior e maior variância.""")

In [ ]:
# ============================================================
# 4.5 AVALIAÇÃO NO CONJUNTO DE TESTE
# ============================================================

modelos_dict = {'Logistic Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gb}
resultados   = {}

for nome, modelo in modelos_dict.items():
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    resultados[nome] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
        'PR-AUC':    average_precision_score(y_test, y_prob),
        'Recall':    recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'y_pred': y_pred, 'y_prob': y_prob,
    }
    print("=" * 60)
    print(f"  {nome}")
    print("=" * 60)
    print(f"  F1-Score:  {resultados[nome]['F1-Score']:.4f}  ← MÉTRICA PRINCIPAL")
    print(f"  PR-AUC:    {resultados[nome]['PR-AUC']:.4f}  ← mais honesta com desbalanceamento")
    print(f"  ROC-AUC:   {resultados[nome]['ROC-AUC']:.4f}")
    print(f"  Recall:    {resultados[nome]['Recall']:.4f}   (% vinhos Alta detectados)")
    print(f"  Precision: {resultados[nome]['Precision']:.4f}   (% previsões Alta corretas)")
    print(f"  Accuracy:  {resultados[nome]['Accuracy']:.4f}   (enganosa com desbalanceamento 6:1)\n")
    print(classification_report(y_test, y_pred, target_names=['Baixa/Média (0)', 'Alta (1)']))

melhor_f1    = max(resultados, key=lambda m: resultados[m]['F1-Score'])
melhor_prauc = max(resultados, key=lambda m: resultados[m]['PR-AUC'])
melhor_rec   = max(resultados, key=lambda m: resultados[m]['Recall'])
baseline_pr  = float(y_test.mean())

print("─" * 65)
print(f"Melhor F1-Score : {melhor_f1} ({resultados[melhor_f1]['F1-Score']:.4f})")
print(f"Melhor PR-AUC   : {melhor_prauc} ({resultados[melhor_prauc]['PR-AUC']:.4f})")
print(f"Melhor Recall   : {melhor_rec} ({resultados[melhor_rec]['Recall']:.4f})")
print(f"Baseline PR-AUC : {baseline_pr:.4f} (modelo aleatório)")

In [ ]:
# ============================================================
# 4.6 GRÁFICO — Comparação de Métricas entre Modelos
# ============================================================
metricas = ['F1-Score', 'PR-AUC', 'ROC-AUC', 'Recall', 'Precision', 'Accuracy']
x     = np.arange(len(metricas))
width = 0.22

fig, ax = plt.subplots(figsize=(15, 6), facecolor=BG_FIG)
style_ax(ax)

for i, (nome, cor) in enumerate(zip(nomes, cores)):
    valores = [resultados[nome][m] for m in metricas]
    bars    = ax.bar(x + i * width, valores, width, label=nome,
                     color=cor, alpha=0.85, edgecolor='white')
    ax.bar_label(bars, fmt='%.2f', fontsize=11, fontweight='bold', color=TEXT, padding=2)

ax.set_xticks(x + width)
ax.set_xticklabels(metricas, fontsize=13, color=TEXT)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Valor da Métrica', fontsize=13, color=TEXT)
ax.set_title('Comparação de Métricas — Conjunto de Teste (distribuição real)',
             fontsize=14, fontweight='bold', color=TEXT, pad=12)
ax.legend(facecolor='#eeeeee', labelcolor=TEXT, fontsize=12)
ax.axhline(baseline_pr, color='#cc6600', linewidth=1.5, linestyle=':',
           label=f'Baseline PR-AUC = {baseline_pr:.2f}')
ax.text(len(metricas) - 0.3, baseline_pr + 0.025,
        f'Baseline PR-AUC = {baseline_pr:.2f}', color='#cc6600', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/06_comparacao_metricas.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

In [ ]:
# ============================================================
# 4.7 CURVAS ROC E PRECISION-RECALL
# ============================================================
# PR é mais informativa com desbalanceamento — foca na classe minoritária (Alta).

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG_FIG)

for nome, cor in zip(nomes, cores):
    fpr, tpr, _ = roc_curve(y_test, resultados[nome]['y_prob'])
    ax_roc.plot(fpr, tpr, color=cor, linewidth=2.5,
                label=f"{nome}  (AUC = {resultados[nome]['ROC-AUC']:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, resultados[nome]['y_prob'])
    ax_pr.plot(rec, prec, color=cor, linewidth=2.5,
               label=f"{nome}  (PR-AUC = {resultados[nome]['PR-AUC']:.3f})")

ax_roc.plot([0, 1], [0, 1], color=C_SPINE, linewidth=1.2, linestyle='--', label='Aleatório (AUC=0.500)')
ax_roc.set(xlabel='Taxa de Falsos Positivos', ylabel='Taxa de Verdadeiros Positivos (Recall)',
           title='Curvas ROC')
ax_roc.legend(facecolor='#eeeeee', labelcolor=TEXT, fontsize=12)
style_ax(ax_roc)

ax_pr.axhline(baseline_pr, color=C_SPINE, linewidth=1.2, linestyle='--',
              label=f'Aleatório (PR-AUC ≈ {baseline_pr:.3f})')
ax_pr.set(xlabel='Recall (Sensibilidade)', ylabel='Precision (Valor Preditivo Positivo)',
          title='Curvas Precision-Recall\n(mais informativa com desbalanceamento)')
ax_pr.legend(facecolor='#eeeeee', labelcolor=TEXT, fontsize=12)
style_ax(ax_pr)

plt.suptitle('Curvas ROC e Precision-Recall — Comparação entre Modelos',
             fontsize=15, fontweight='bold', color=TEXT, y=1.02)
plt.tight_layout()
plt.savefig('../results/07_curvas_roc_pr.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

# Interpretação dinâmica — sem valores hardcoded
print("CURVA ROC:")
for nome in nomes:
    print(f"  {nome}: AUC = {resultados[nome]['ROC-AUC']:.3f}")
print(f"\nCURVA PR (mais confiável com desbalanceamento):")
for nome in nomes:
    ganho = resultados[nome]['PR-AUC'] / baseline_pr
    print(f"  {nome}: PR-AUC = {resultados[nome]['PR-AUC']:.3f}  ({ganho:.1f}x acima do baseline)")

In [ ]:
# ============================================================
# 4.8 MATRIZES DE CONFUSÃO
# ============================================================
# FN = vinho premium → classificado como comum → perda de receita (erro mais caro)
# FP = vinho comum  → classificado como premium → cliente insatisfeito

fig, axes = plt.subplots(1, 3, figsize=(17, 6), facecolor=BG_FIG)
rotulos = [['VN', 'FP'], ['FN', 'VP']]

for ax, (nome, cor) in zip(axes, zip(nomes, cores)):
    cm  = confusion_matrix(y_test, resultados[nome]['y_pred'])
    pct = cm / cm.sum() * 100
    sns.heatmap(cm, annot=False, ax=ax, cmap=sns.light_palette(cor, as_cmap=True),
                linewidths=2, linecolor='white', cbar=False)
    threshold = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            cor_t = 'white' if cm[i, j] > threshold else '#222222'
            ax.text(j + 0.5, i + 0.5, f'{rotulos[i][j]}\n{cm[i,j]}\n({pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=13, fontweight='bold', color=cor_t)
    ax.set_xticklabels(['Previu\nBaixa/Média', 'Previu\nAlta'], color=TEXT, fontsize=12)
    ax.set_yticklabels(['Real\nBaixa/Média', 'Real\nAlta'],     color=TEXT, fontsize=12, rotation=0)
    ax.set_title(nome, fontsize=13, fontweight='bold', color=TEXT, pad=10)

plt.suptitle('Matrizes de Confusão — Conjunto de Teste\n'
             '(VN/VP = acertos | FP = alarme falso | FN = miss — erro mais custoso)',
             fontsize=13, fontweight='bold', color=TEXT, y=1.03)
plt.tight_layout()
plt.savefig('../results/08_matrizes_confusao.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

In [ ]:
# ============================================================
# 4.9 FEATURE IMPORTANCE — Gini + Permutation (RF e GB)
# ============================================================
# GINI: redução de impureza nas árvores (pode superestimar features contínuas).
# PERMUTATION: embaralha feature e mede queda no F1 → impacto REAL na métrica.
# Convergência Gini ↔ Permutation = resultado confiável.

fig, axes = plt.subplots(2, 2, figsize=(18, 14), facecolor=BG_FIG)

for row_idx, (modelo_nome, modelo_obj) in enumerate([('Random Forest', rf), ('Gradient Boosting', gb)]):
    imp_gini = pd.Series(modelo_obj.feature_importances_, index=features_model).sort_values(ascending=True)
    pi       = permutation_importance(modelo_obj, X_test, y_test, n_repeats=15, random_state=42, scoring='f1')
    imp_perm = pd.Series(pi.importances_mean, index=features_model).sort_values(ascending=True)

    for col_idx, (imp, title) in enumerate([
        (imp_gini, f'{modelo_nome}\nGini Importance (redução de impureza)'),
        (imp_perm, f'{modelo_nome}\nPermutation Importance (impacto no F1)')
    ]):
        ax = axes[row_idx][col_idx]
        style_ax(ax)
        cores_imp = [C_HIGH if i >= len(imp)-5 else C_LOW for i in range(len(imp))]
        bars = ax.barh(imp.index, imp.values, color=cores_imp, edgecolor='white')
        ax.bar_label(bars, fmt='%.3f', label_type='center', color='white',
                     fontsize=11, fontweight='bold', padding=0,
                     bbox=dict(boxstyle='round,pad=0.4', facecolor='black', alpha=0.4, edgecolor='none'))
        ax.set_xlabel('Importância', fontsize=13, color=TEXT)
        ax.set_title(title, fontsize=13, fontweight='bold', color=TEXT, pad=10)
        ax.axvline(0, color=C_SPINE, linewidth=0.8)

plt.suptitle('Feature Importance — RF e GB (verde = top 5)\nGini vs Permutation',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig('../results/09_feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

print("Convergência entre Gini e Permutation = feature realmente importante.")
print("Divergência = Gini pode estar superestimando (comum em features contínuas).")

---
# ETAPA 5 — Interpretação dos Resultados

## 5.1 Escolha do Modelo Vencedor

**Gradient Boosting** lidera em **F1-Score** e **Recall** — melhor quando o custo de
perder um vinho premium (falso negativo) é maior que classificar um vinho comum como premium.

**Random Forest** lidera em **PR-AUC** — escolha quando se prioriza confiabilidade geral.

**Logistic Regression** confirma que separação linear é insuficiente para este problema.

> **Decisão:** Gradient Boosting como modelo principal, Random Forest como alternativo.

In [ ]:
# ============================================================
# 5.2 TABELA COMPARATIVA FINAL
# ============================================================
print("=" * 85)
print(f"  {'COMPARAÇÃO FINAL DOS MODELOS':^81}")
print("=" * 85)
print(f"{'Modelo':<25} {'F1':>8} {'PR-AUC':>8} {'ROC-AUC':>9} {'Recall':>8} {'Precision':>10} {'Accuracy':>10}")
print("─" * 85)

for nome in nomes:
    r = resultados[nome]
    destaque = ""
    if nome == melhor_f1:    destaque += " ← melhor F1"
    if nome == melhor_prauc: destaque += " / melhor PR-AUC"
    if nome == melhor_rec:   destaque += " / melhor Recall"
    print(f"{nome:<25} {r['F1-Score']:>8.4f} {r['PR-AUC']:>8.4f}"
          f" {r['ROC-AUC']:>9.4f} {r['Recall']:>8.4f} {r['Precision']:>10.4f}"
          f" {r['Accuracy']:>10.4f}{destaque}")

print("─" * 85)
print(f"\n  Baseline PR-AUC (modelo aleatório): {baseline_pr:.4f}")
for nome in nomes:
    print(f"  {nome:<25}: {resultados[nome]['PR-AUC']/baseline_pr:.1f}x acima do baseline")

In [ ]:
# ============================================================
# 5.3 THRESHOLD TUNING — Otimizando o limiar de decisão
# ============================================================
# predict() usa threshold=0.5 por padrão.
# Com desbalanceamento 6:1, ajustar o threshold pode melhorar F1.
# Threshold baixo → Recall ↑ Precision ↓ | Threshold alto → Recall ↓ Precision ↑

thresholds  = np.arange(0.10, 0.91, 0.01)
y_prob_gb   = resultados['Gradient Boosting']['y_prob']

f1_list   = [(y_prob_gb >= t).astype(int) for t in thresholds]
f1_scores  = [f1_score(y_test, p, zero_division=0)        for p in f1_list]
prec_scores = [precision_score(y_test, p, zero_division=0) for p in f1_list]
rec_scores  = [recall_score(y_test, p,  zero_division=0)   for p in f1_list]

best_idx    = int(np.argmax(f1_scores))
best_thresh = float(thresholds[best_idx])
best_f1     = float(f1_scores[best_idx])
best_prec   = float(prec_scores[best_idx])
best_rec    = float(rec_scores[best_idx])

fig, ax = plt.subplots(figsize=(13, 6), facecolor=BG_FIG)
style_ax(ax)
ax.plot(thresholds, f1_scores,   color=C_GB,  linewidth=2.5, label='F1-Score')
ax.plot(thresholds, prec_scores, color=C_LOW,  linewidth=1.8, linestyle='--', label='Precision')
ax.plot(thresholds, rec_scores,  color=C_HIGH, linewidth=1.8, linestyle='--', label='Recall')
ax.axvline(0.5,         color='#888888', linewidth=1.2, linestyle=':', label='Threshold padrão (0.50)')
ax.axvline(best_thresh, color=C_GB,      linewidth=2.0, linestyle='-.', label=f'Melhor F1 (threshold={best_thresh:.2f})')
ax.scatter([best_thresh], [best_f1], color=C_GB, zorder=5, s=80)
ax.annotate(f'F1={best_f1:.3f}\nP={best_prec:.3f}\nR={best_rec:.3f}',
            xy=(best_thresh, best_f1), xytext=(best_thresh + 0.06, best_f1 - 0.10),
            fontsize=11, color=TEXT, arrowprops=dict(arrowstyle='->', color=C_SPINE))
ax.set_xlabel('Threshold de decisão', fontsize=13, color=TEXT)
ax.set_ylabel('Valor da métrica', fontsize=13, color=TEXT)
ax.set_title('Gradient Boosting — F1, Precision e Recall por Threshold\n'
             '(threshold ótimo = máximo F1)', fontsize=14, fontweight='bold', color=TEXT)
ax.legend(fontsize=11, facecolor='#eeeeee', labelcolor=TEXT)
plt.tight_layout()
plt.savefig('../results/10_threshold_tuning.png', dpi=150, bbox_inches='tight', facecolor=BG_FIG)
plt.show()

f1_padrao = resultados['Gradient Boosting']['F1-Score']
print("=" * 65)
print("  THRESHOLD 0.50 vs THRESHOLD ÓTIMO — Gradient Boosting")
print("=" * 65)
print(f"  Threshold padrão  (0.50): F1 = {f1_padrao:.4f}")
print(f"  Threshold ótimo ({best_thresh:.2f}): F1 = {best_f1:.4f}  (+{best_f1 - f1_padrao:.4f})")
print(f"  Precision no threshold ótimo: {best_prec:.4f}")
print(f"  Recall    no threshold ótimo: {best_rec:.4f}")
print(f"\n  → Use threshold {best_thresh:.2f} em produção para maximizar F1.")
print("  → Para priorizar Recall (evitar FN), use threshold menor.")
print("=" * 65)

In [ ]:
# ============================================================
# 5.4 ANÁLISE DAS VARIÁVEIS MAIS INFLUENTES — Top 5 Features
# ============================================================
imp_gini_rf = pd.Series(rf.feature_importances_, index=features_model).sort_values(ascending=False)
pi_rf       = permutation_importance(rf, X_test, y_test, n_repeats=15, random_state=42, scoring='f1')
imp_perm_rf = pd.Series(pi_rf.importances_mean, index=features_model).sort_values(ascending=False)

top5_gini = imp_gini_rf.head(5).index.tolist()
top5_perm = imp_perm_rf.head(5).index.tolist()

print("Top 5 Features — Gini Importance (RF):")
for i, feat in enumerate(top5_gini, 1):
    print(f"  {i}. {feat:<28}: {imp_gini_rf[feat]:.4f}")

print("\nTop 5 Features — Permutation Importance (RF):")
for i, feat in enumerate(top5_perm, 1):
    print(f"  {i}. {feat:<28}: {imp_perm_rf[feat]:.4f}")

print("""
─────────────────────────────────────────────────────────
IMPLICAÇÕES PARA O NEGÓCIO
─────────────────────────────────────────────────────────
→ alcohol / alcohol_density:
   Maior teor alcoólico = maior qualidade.
   AÇÃO: monitorar temperatura e tempo de fermentação
   para maximizar conversão de açúcar em álcool.

→ volatile acidity:
   Excesso de ácido acético = principal fator negativo.
   AÇÃO: controle microbiológico rigoroso (temperatura,
   limpeza) para prevenir contaminação bacteriana.

→ sulphates:
   Mais sulfatos = melhor conservação antimicrobiana.
   AÇÃO: calibrar adição de sulfatos na vinificação.

→ citric acid / acid_ratio:
   Equilíbrio ácido importa mais que valores absolutos.
   AÇÃO: ajustar acidez cítrica para frescor e complexidade.
─────────────────────────────────────────────────────────""")

---
# ETAPA 6 — Conclusões e Recomendações

## 6.1 Síntese dos Resultados

| Ponto | Resultado |
|---|---|
| Dataset | 1.018 amostras (após remoção de duplicatas), 11 features originais + 3 engineered |
| Desbalanceamento | 6:1 → tratado com SMOTE via ImbPipeline (sem data leakage) |
| Modelo vencedor (F1/Recall) | **Gradient Boosting** |
| Modelo alternativo (PR-AUC) | **Random Forest** |
| Ganho sobre baseline aleatório | ~4-5x em PR-AUC |
| Features mais importantes | `alcohol`, `alcohol_density`, `volatile acidity` |

## 6.2 Os 4 Fatores que Mais Distinguem Vinhos de Alta Qualidade

1. **Teor alcoólico (alcohol)** — principal preditor positivo; controlável via fermentação
2. **Acidez volátil (volatile acidity)** — principal preditor negativo; controle microbiológico
3. **Sulfatos (sulphates)** — correlação positiva; proxy de conservação
4. **Ácido cítrico / acid_ratio** — equilíbrio entre tipos de acidez

## 6.3 Limitações

- **Dataset pequeno (1.018 amostras):** notas extremas (3, 4, 8) são raras → modelo menos confiável nesses casos
- **Apenas vinho tinto:** não aplicável diretamente para vinho branco
- **Threshold ajustado:** o threshold ótimo foi identificado via tuning (Etapa 5.3) e deve ser atualizado se o modelo for retreinado com novos dados

## 6.4 Próximos Passos Recomendados

1. **Otimização de hiperparâmetros** via Optuna para refinar os modelos
2. **Calibração de probabilidades** (Platt Scaling) para melhor uso do threshold
3. **Coletar mais dados** de vinhos com notas extremas (3, 4, 8)
4. **Testar em vinho branco** com retraining específico

## 6.5 Aplicação Prática

O modelo pode ser integrado ao processo produtivo como **sistema de alerta precoce**:
medir variáveis físico-químicas durante a produção e receber predição de qualidade
**antes da avaliação sensorial final** — permitindo ajustes em tempo real.

In [ ]:
# ============================================================
# VERSÕES DAS BIBLIOTECAS UTILIZADAS
# ============================================================
import pkg_resources

libs = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy',
    'scikit-learn', 'imbalanced-learn'
]
print(f"{'Biblioteca':<20} {'Versão'}")
print("─" * 35)
for lib in libs:
    try:
        v = pkg_resources.get_distribution(lib).version
        print(f"{lib:<20} {v}")
    except Exception:
        print(f"{lib:<20} não encontrada")

print("\nEste output deve ser usado para gerar/atualizar o requirements.txt do projeto.")